In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.float_format', lambda x: '%.2f' % x)

def contar_nulos(df, columna):
    """
    Cuenta y muestra los valores faltantes en una columna 
    """
    total_nulos = df[columna].isnull().sum()
    print(f"Valores faltantes en {columna}: {total_nulos}")
    return total_nulos

def analizar_columna(df, columna):
    """
    Muestra estadísticas descriptivas y conteo de valores para una columna
    """
    print(f"--- Resumen de: {columna} ---")
    print("\n📊 Estadísticas Descriptivas:")
    print(df[columna].describe())
    
    print("\n🔢 Conteo de Valores (Frecuencias):")
    print(df[columna].value_counts())
    print("-" * 30)

def analizar(df,columna):
    analizar_columna(df, columna)
    contar_nulos(df,columna)


In [2]:
df = pd.read_csv("../data/interim/mcq_h_columnas.csv")
df

,SEQN,MCQ160M,MCQ160L,MCQ160A,MCQ160F,MCQ160E,MCQ160C,MCQ220,MCQ010,MCQ080,MCQ070
0,73557.00,2.00,2.00,1.00,1.00,2.00,2.00,2.00,2.00,1.00,2.00
1,73558.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,1.00,2.00,2.00
2,73559.00,2.00,2.00,2.00,2.00,2.00,2.00,1.00,2.00,2.00,2.00
3,73560.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.00,NaN,NaN
4,73561.00,1.00,2.00,1.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00
...,...,...,...,...,...,...,...,...,...,...,...
9765,83727.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,1.00,2.00,2.00
9766,83728.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.00,NaN,NaN
9767,83729.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00
9768,83730.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.00,NaN,NaN


In [3]:
# Filtrar solo mayores a 18
df_demo = pd.read_csv("../data/interim/demo_h_columnas.csv")
df_edad = df_demo[['SEQN', 'RIDAGEYR']].copy()
df = df.merge(df_edad,on='SEQN',how='inner')
df = df[df['RIDAGEYR']>=18]

In [4]:
renombrar_condiciones_medicas = {
    "MCQ160M": "problema_tiroides",
    "MCQ160L": "problema_higado",
    "MCQ160A": "tiene_artritis",
    "MCQ160F": "tuvo_derrame_cerebral",
    "MCQ160E": "tuvo_ataque_corazon",
    "MCQ160C": "enfermedad_coronaria",
    "MCQ220": "tuvo_cancer",
    "MCQ010": "tiene_asma",
    "MCQ080": "tiene_sobrepeso",
    "MCQ070": "tiene_psoriasis"
}

# Para aplicarlo a tu DataFrame:
df = df.rename(columns=renombrar_condiciones_medicas)

In [5]:
nulos_por_columna = df.isna().sum()
print("Cantidad de nulos por columna:")
print(nulos_por_columna)

Cantidad de nulos por columna:
SEQN                       0
problema_tiroides        344
problema_higado          344
tiene_artritis           344
tuvo_derrame_cerebral    344
tuvo_ataque_corazon      344
enfermedad_coronaria     344
tuvo_cancer              344
tiene_asma                 0
tiene_sobrepeso            0
tiene_psoriasis            0
RIDAGEYR                   0
dtype: int64


In [6]:
cols_medicas = [
    "problema_tiroides",
    "problema_higado",
    "tiene_artritis",
    "tuvo_derrame_cerebral",
    "tuvo_ataque_corazon",
    "enfermedad_coronaria",
    "tuvo_cancer",
    "tiene_asma",
    "tiene_sobrepeso",
    "tiene_psoriasis"
]

for col in cols_medicas:
    analizar(df,col)

--- Resumen de: problema_tiroides ---

📊 Estadísticas Descriptivas:
count   5769.00
mean       1.91
std        0.44
min        1.00
25%        2.00
50%        2.00
75%        2.00
max        9.00
Name: problema_tiroides, dtype: float64

🔢 Conteo de Valores (Frecuencias):
problema_tiroides
2.00    5157
1.00     601
9.00      11
Name: count, dtype: int64
------------------------------
Valores faltantes en problema_tiroides: 344
--- Resumen de: problema_higado ---

📊 Estadísticas Descriptivas:
count   5769.00
mean       1.97
std        0.35
min        1.00
25%        2.00
50%        2.00
75%        2.00
max        9.00
Name: problema_higado, dtype: float64

🔢 Conteo de Valores (Frecuencias):
problema_higado
2.00    5525
1.00     234
9.00      10
Name: count, dtype: int64
------------------------------
Valores faltantes en problema_higado: 344
--- Resumen de: tiene_artritis ---

📊 Estadísticas Descriptivas:
count   5769.00
mean       1.75
std        0.56
min        1.00
25%        1.00
50%

In [7]:

# Los 7 (Refused), 9 (Don't Know) y los vacíos originales se vuelven NaN.
for col in cols_medicas:
    df[col] = df[col].map({1: True, 2: False})

# Rellenar todos los NaN con False
# jóvenes de 18-19 años.
df[cols_medicas] = df[cols_medicas].fillna(False)

# 3. Verificamos que ya no queden nulos
print("Nulos restantes en columnas médicas:")
print(df[cols_medicas].isna().sum())

Nulos restantes en columnas médicas:
problema_tiroides        0
problema_higado          0
tiene_artritis           0
tuvo_derrame_cerebral    0
tuvo_ataque_corazon      0
enfermedad_coronaria     0
tuvo_cancer              0
tiene_asma               0
tiene_sobrepeso          0
tiene_psoriasis          0
dtype: int64


/tmp/ipykernel_8219/3680267735.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[cols_medicas] = df[cols_medicas].fillna(False)


In [8]:
df

,SEQN,problema_tiroides,problema_higado,tiene_artritis,tuvo_derrame_cerebral,tuvo_ataque_corazon,enfermedad_coronaria,tuvo_cancer,tiene_asma,tiene_sobrepeso,tiene_psoriasis,RIDAGEYR
0,73557.00,False,False,True,True,False,False,False,False,True,False,69.00
1,73558.00,False,False,False,False,False,False,False,True,False,False,54.00
2,73559.00,False,False,False,False,False,False,True,False,False,False,72.00
4,73561.00,True,False,True,False,False,False,False,False,False,False,73.00
5,73562.00,True,False,True,False,True,True,False,False,True,False,56.00
...,...,...,...,...,...,...,...,...,...,...,...,...
9761,83723.00,False,False,False,False,False,False,False,False,False,False,61.00
9762,83724.00,False,False,True,False,False,False,True,False,False,False,80.00
9764,83726.00,False,False,False,False,False,False,False,False,False,False,40.00
9765,83727.00,False,False,False,False,False,False,False,True,False,False,26.00


In [9]:
df = df.drop(columns=['RIDAGEYR'])

df.to_csv('../data/mcq.csv', index=False)

### problema_tiroides

In [10]:
analizar(df,'problema_tiroides')

--- Resumen de: problema_tiroides ---

📊 Estadísticas Descriptivas:
count      6113
unique        2
top       False
freq       5512
Name: problema_tiroides, dtype: object

🔢 Conteo de Valores (Frecuencias):
problema_tiroides
False    5512
True      601
Name: count, dtype: int64
------------------------------
Valores faltantes en problema_tiroides: 0


### problema_higado

In [11]:
analizar(df,'problema_higado')

--- Resumen de: problema_higado ---

📊 Estadísticas Descriptivas:
count      6113
unique        2
top       False
freq       5879
Name: problema_higado, dtype: object

🔢 Conteo de Valores (Frecuencias):
problema_higado
False    5879
True      234
Name: count, dtype: int64
------------------------------
Valores faltantes en problema_higado: 0


### tiene_artritis

### tuvo_derrame_cerebral

### tuvo_ataque_corazon

### enfermedad_coronaria

### tuvo_cancer

### tiene_asma

### tiene_sobrepeso

### tiene_psoriasis